In [2]:
import time
from datetime import datetime
import os
import csv
import scipy.constants as cs
from RsInstrument import RsInstrument
import numpy as np

#### Time control notes
The total time we need to account for between trials is: 
- Time for polarization (300ns) https://arxiv.org/pdf/1809.05237
- Time for $\pi/2$ pulse * 2
- Chosen wait time $\tau$ * 2 [We vary this to get datapoints to fit]
- Time for $\pi$ pulse

Then during the readout stage (optical counting duration at 225 ns) https://arxiv.org/pdf/1809.05237

### Calculating the duration of our $pi/2$ pulse:
we're going to use 
$t_{\pi/2} = \frac{\pi}{2*\gamma*B_1}$ 

where $\gamma$ is the gyromagnetic ratio for NV centers and $B_1$ is the applied magnetic field from the microwave coil.
we define $\gamma = \frac{q*g_n}{2*m_p}$ from here: https://web.physics.ucsb.edu/~phys128/experiments/nmr/Pulsed_NMR.pdf

where $q$ is the elementary charge of an electron, $m_p$ is the mass of a proton$, and $g_n$ is the g-factor associated with the NV diamond. In this case, $g=2.0023$, from this paper: https://iopscience.iop.org/article/10.1088/0034-4885/41/8/002/pdf

In [10]:
g = 2.0023
gyro_ratio = g * cs.e / (2 * cs.m_p)
microwave_field = 3e-4 # Tesla
t_p2 = cs.pi / (2*gyro_ratio * microwave_field)
t_p = t_p2 * 2
print(f"t_p2 = {t_p2:.3e} s")
print(f"t_p = {t_p:.3e} s")
print(f"Total time for one experiment with no wait time = {300e-9 + 225e-9 + 2 * t_p2 + t_p:.3e} s")
print(1/(300e-9 + 225e-9 + 2 * t_p2 + t_p), "Hz")

t_p2 = 5.460e-05 s
t_p = 1.092e-04 s
Total time for one experiment with no wait time = 2.189e-04 s
4567.835001858139 Hz


In [11]:
def time_calc(tao, reptitions=10000):
    total_time = (300e-9 + 225e-9 + 2 * t_p2 + t_p + 2 * tao + 100e-9)*reptitions # 100 ns wait time between experiments
    # print(f"Total time excluding wait time = {total_time:.3e} s")
    return total_time

tao_points_estimate = np.arange(0, 400e-6, 5e-6) # start, stop, increment length
total_times_estimate = (300e-9 + 225e-9 + 2 * t_p2 + t_p + 2 * tao_points_estimate + 100e-9) * 100
print("total time for all data collection:", np.sum(total_times_estimate), "s")

total time for all data collection: 4.912176745601735 s


In [ ]:
def archive_scope_waveform(resource_string, destination_folder, t_1, t_2, rep, partition):
    # resource_string = address of the scope
    # destination_folder = where the csv files will be saved
    # t_1 = starting wait time (tao) for experiment  (must be greater than 0)
    # t_2 = ending wait time (tao) (must be greater than t_1)
    # rep = number of trials before an average is recorded for the given setup's tao value (i.e., for each tao value, we record 10e5 trials to reduce variance)
    # partition = number of steps from t_1 to t_2, must be greater than 0


    scope = RsInstrument(resource_string, id_query=True, reset=False)

    scope.write_str('*CLS') 
    scope.write_str('CHAN1:STAT ON') #channel 1 status on
    scope.write_str('FORM REAL') 
    tao_points = np.arange(t_1, t_2, partition) # start, stop, increment length
    total_times = (300e-9 + 225e-9 + 2 * t_p2 + t_p + 2 * tao_points + 100e-9) # adding 100ns delay between experiments for program to catch up

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S') #naming of the csv file with the exact date/time
    csv_filename = os.path.join(destination_folder, f'data_{timestamp}.csv')

    with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Time (s)', 'Voltage (V)'])
        tao_step = 0
        for times in total_times:
            # Trigger a single acquisition: this is taking the "screenshot"
            min_voltage = -99999
            while rep > 0:
                scope.write_str('SINGLE')
                scope.query_opc()

                # Read waveform data in binary (float32)
                binary_data = scope.query_bin_block('CHAN1:DATA?')

                y_inc = float(scope.query_str('CHAN:DATA:YINC?')) 
                y_zero = float(scope.query_str('CHAN:DATA:YOR?'))
                scaled_voltage = y_zero + binary_data[1] * y_inc

                if np.min(scaled_voltage) >= min_voltage:
                    min_voltage = np.max(scaled_voltage)
                time.sleep(times)
                rep -= 1

            # Only recording the minimum voltage observed after 225ns observation window, corresponding to a dip in the measured red fluorescence coming from states in the metastable state 
            writer.writerow([tao_points[tao_step], min_voltage])
            tao_step += 1
        csvfile.close()
    scope.close()



In [8]:
if __name__ == '__main__':
    visa_resource = 'USB0::0x0AAD::0x01D6::108904::INSTR'  # the address of the oscilloscope
    destination = r"C:\Users\allen\Desktop\Homework\Physics 108\Scope"
    start_tao = 0
    end_tao = 2
    repeats = 1
    partitions = 0.1
    archive_scope_waveform(visa_resource, destination, start_tao, end_tao, repeats, partitions) 